# 🏨 NaraHoteis — Consultoria de Dados
## Diagnóstico e Painel Gerencial

**Cliente:** NaraHoteis — Rede Hoteleira do Estado do Rio de Janeiro  
**Documento de referência:** COM-2025-047 (Comunicado Oficial do Departamento de TI)  
**Guia utilizado:** Guia de Boas Práticas — Tratamento de Dados (NaraHoteis)

---

### Sumário

1. [Importações e Carregamento das Bases](#1-importações-e-carregamento-das-bases)
2. [Auditoria Geral](#2-auditoria-geral)
3. [Pré-processamento — reservas.csv](#3-pré-processamento--reservascsv)
4. [Pré-processamento — unidades.csv](#4-pré-processamento--unidadescsv)
5. [Pré-processamento — tipos_quarto.csv](#5-pré-processamento--tipos_quartoscsv)
6. [Pré-processamento — clientes.csv](#6-pré-processamento--clientescsv)
7. [Pré-processamento — canais_venda.csv](#7-pré-processamento--canais_vendacsv)
8. [Pré-processamento — funcionarios.csv](#8-pré-processamento--funcionarioscsv)
9. [Exportação das Bases Tratadas](#9-exportação-das-bases-tratadas)
10. [Análise Estatística](#10-análise-estatística)
11. [Painel — Matplotlib](#11-painel--matplotlib)


---
## 1. Importações e Carregamento das Bases

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Carregamento das 6 bases brutas
reservas     = pd.read_csv('../dados/brutos/reservas.csv', sep=';')
unidades     = pd.read_csv('../dados/brutos/unidades.csv', sep=';')
tipos_quarto = pd.read_csv('../dados/brutos/tipos_quarto.csv', sep=';')
clientes     = pd.read_csv('../dados/brutos/clientes.csv', sep=';')
canais_venda = pd.read_csv('../dados/brutos/canais_venda.csv', sep=';')
funcionarios = pd.read_csv('../dados/brutos/funcionarios.csv', sep=';')

---
## 2. Auditoria Geral

Antes de qualquer tratamento, realizamos uma auditoria inicial em todas as bases para mapear o volume de dados, tipos de colunas e presença de valores nulos.

In [ ]:
bases = {
    'reservas': reservas,
    'unidades': unidades,
    'tipos_quarto': tipos_quarto,
    'clientes': clientes,
    'canais_venda': canais_venda,
    'funcionarios': funcionarios,
}

for nome, df in bases.items():
    print(f"{'='*50}")
    print(f" {nome.upper()}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape}")
    print(f"\nTipos:\n{df.dtypes}")
    print(f"\nNulos:\n{df.isnull().sum()}")
    print()

---
## 3. Pré-processamento — reservas.csv

### 3.1 Visão inicial

In [ ]:
reservas.head(10)

In [ ]:
reservas.describe(include='all')

### 3.2 Campo: `qtd_diarias`

**🔍 Problema identificado:** Valores impossíveis — `0` e `-1` foram encontrados neste campo.  
Uma reserva não pode ter zero ou número negativo de diárias operacionalmente.

**✅ Ação:** Substituição pelos valores corretos conforme **Comunicado Oficial COM-2025-047**.

**📌 Boa prática:** Valores operacionalmente impossíveis devem ser corrigidos com base em fonte oficial, nunca estimados ou removidos sem justificativa.

In [ ]:
# Identificando registros com qtd_diarias inválida
reservas[reservas['qtd_diarias'] <= 0][['id_reserva', 'qtd_diarias']]

In [ ]:
# Correção com base no Comunicado Oficial COM-2025-047
correcoes_qtd = {
    728: 4, 955: 4, 1134: 2, 1152: 5,
    1448: 2, 1581: 2, 2312: 2, 2384: 2
}

for id_reserva, valor_correto in correcoes_qtd.items():
    reservas.loc[reservas['id_reserva'] == id_reserva, 'qtd_diarias'] = valor_correto

# Verificação
print("Registros com qtd_diarias <= 0 após correção:")
print(reservas[reservas['qtd_diarias'] <= 0].shape[0])

### 3.3 Campo: `avaliacao_hospede`

**🔍 Problema identificado:** Valores fora da escala permitida — encontrados `0`, `11` e `-3`.  
A escala de avaliação da NaraHoteis é de **1 a 10**.  
Registros com valor **nulo** são válidos — representam hóspedes que optaram por não avaliar.

**✅ Ação:** Substituição pelos valores corretos conforme **Comunicado Oficial COM-2025-047**.

**📌 Boa prática:** Sempre documente a escala esperada de cada campo numérico antes de iniciar o tratamento. Valores fora da escala são erros de entrada, não outliers estatísticos.

In [ ]:
# Identificando avaliações fora da escala (excluindo nulos que são válidos)
mask_invalida = reservas['avaliacao_hospede'].notna() & ~reservas['avaliacao_hospede'].between(1, 10)
reservas[mask_invalida][['id_reserva', 'avaliacao_hospede']]

In [ ]:
# Correção com base no Comunicado Oficial COM-2025-047
correcoes_aval = {
    37: 8, 194: 2, 902: 10, 933: 7,
    1538: 3, 1718: 4, 1866: 3, 2073: 9, 2167: 5, 2210: 2
}

for id_reserva, valor_correto in correcoes_aval.items():
    reservas.loc[reservas['id_reserva'] == id_reserva, 'avaliacao_hospede'] = valor_correto

# Verificação
mask_invalida = reservas['avaliacao_hospede'].notna() & ~reservas['avaliacao_hospede'].between(1, 10)
print(f"Avaliações fora da escala após correção: {mask_invalida.sum()}")
print(f"Avaliações nulas (válidas — hóspede não avaliou): {reservas['avaliacao_hospede'].isna().sum()}")

### 3.4 Campo: `num_hospedes`

**🔍 Problema identificado:** Valores negativos — `-1` e `-2` foram encontrados.  
O número de hóspedes é uma contagem — não pode ser negativo.

**✅ Ação:** Substituição pelos valores corretos conforme **Comunicado Oficial COM-2025-047**.

**📌 Boa prática:** Campos de contagem jamais podem ser negativos. Identifique e corrija antes de qualquer análise.

In [ ]:
# Identificando num_hospedes negativos
reservas[reservas['num_hospedes'] < 0][['id_reserva', 'num_hospedes']]

In [ ]:
# Correção com base no Comunicado Oficial COM-2025-047
correcoes_hosp = {
    568: 2, 675: 3, 903: 2,
    1201: 3, 1883: 1, 2427: 1
}

for id_reserva, valor_correto in correcoes_hosp.items():
    reservas.loc[reservas['id_reserva'] == id_reserva, 'num_hospedes'] = valor_correto

print(f"Registros com num_hospedes < 0 após correção: {(reservas['num_hospedes'] < 0).sum()}")

### 3.5 Campo: `status_reserva`

**🔍 Problema identificado:** Variações de escrita para o mesmo valor — `'confirmada'`, `'Confirmada'`, `'CONFIRMADA '`, `'conf.'` representam o mesmo status mas estão escritos de formas diferentes.

**✅ Ação:** Padronização para os valores oficiais: `Confirmada`, `Cancelada`, `No-Show`, `Concluída`.

**📌 Boa prática:** Campos categóricos devem ter valores controlados. Padronize caixa, remova espaços extras e unifique abreviações antes de qualquer agrupamento ou filtro.

In [ ]:
# Auditoria dos valores únicos
print("Valores únicos em status_reserva:")
print(reservas['status_reserva'].value_counts())

In [ ]:
# Mapeamento para padronização
mapa_status = {}
for val in reservas['status_reserva'].dropna().unique():
    val_norm = val.strip().lower()
    if val_norm in ['confirmada', 'conf.']:
        mapa_status[val] = 'Confirmada'
    elif val_norm in ['cancelada', 'cancel.']:
        mapa_status[val] = 'Cancelada'
    elif val_norm in ['no-show', 'noshow', 'no show']:
        mapa_status[val] = 'No-Show'
    elif val_norm in ['concluída', 'concluida']:
        mapa_status[val] = 'Concluída'

reservas['status_reserva'] = reservas['status_reserva'].map(mapa_status)

print("Valores após padronização:")
print(reservas['status_reserva'].value_counts())

### 3.6 Campo: `forma_pagamento`

**🔍 Problema identificado:** Abreviações e variações inconsistentes — `'CC'`, `'cartao credito'`, `'Cartão de Crédito'`, `'cred'` representam o mesmo meio de pagamento.

**✅ Ação:** Padronização para os valores oficiais: `Cartão de Crédito`, `Cartão de Débito`, `PIX`, `Dinheiro`, `Transferência`.

**📌 Boa prática:** Campos de forma de pagamento são críticos para análises financeiras. Qualquer inconsistência pode distorcer totais e distribuições.

In [ ]:
print("Valores únicos em forma_pagamento:")
print(reservas['forma_pagamento'].value_counts())

In [ ]:
mapa_pagto = {}
for val in reservas['forma_pagamento'].dropna().unique():
    val_norm = val.strip().lower()
    if any(x in val_norm for x in ['crédito', 'credito', 'cc', 'cred', 'c. cré']):
        mapa_pagto[val] = 'Cartão de Crédito'
    elif any(x in val_norm for x in ['débito', 'debito', 'cd', 'déb']):
        mapa_pagto[val] = 'Cartão de Débito'
    elif 'pix' in val_norm:
        mapa_pagto[val] = 'PIX'
    elif any(x in val_norm for x in ['dinheiro', 'cash']):
        mapa_pagto[val] = 'Dinheiro'
    elif any(x in val_norm for x in ['transfer', 'ted']):
        mapa_pagto[val] = 'Transferência'

reservas['forma_pagamento'] = reservas['forma_pagamento'].map(mapa_pagto)

print("Valores após padronização:")
print(reservas['forma_pagamento'].value_counts())

### 3.7 Campo: `id_canal` — Nulos

**🔍 Problema identificado:** Aproximadamente 5% dos registros apresentam `id_canal` nulo.

**✅ Ação:** Manter os nulos. Estes registros representam reservas antigas realizadas antes da implantação do controle por canal de venda.

**📌 Boa prática:** Nem todo nulo é um erro. Antes de tratar, investigue a origem. Nulos com justificativa de negócio devem ser documentados e mantidos.

In [ ]:
nulos_canal = reservas['id_canal'].isna().sum()
pct = nulos_canal / len(reservas) * 100
print(f"Registros com id_canal nulo: {nulos_canal} ({pct:.1f}%)")
print("Decisão: MANTER — reservas anteriores à implantação do controle por canal.")

### 3.8 Verificação final — reservas.csv

In [ ]:
print("Shape final:", reservas.shape)
print("\nNulos remanescentes:")
print(reservas.isnull().sum())
print("\nTipos:")
print(reservas.dtypes)

---
## 4. Pré-processamento — unidades.csv

In [ ]:
unidades.head()

### 4.1 Campo: `regiao`

**🔍 Problema identificado:** Variações de escrita — `'capital'`, `'CAP'`, `'Cap.'`, `'Baixada fluminense'`, `'Serra '`.

**✅ Ação:** Padronização para os 4 valores oficiais: `Capital`, `Baixada Fluminense`, `Serra`, `Costa Verde`.

**📌 Boa prática:** Campos geográficos são frequentemente usados para agrupamento e filtro. Qualquer variação compromete análises regionais inteiras.

In [ ]:
print("Valores únicos em regiao:")
print(unidades['regiao'].value_counts())

In [ ]:
mapa_regiao = {}
for val in unidades['regiao'].dropna().unique():
    val_norm = val.strip().lower()
    if 'capital' in val_norm or val_norm in ['cap.', 'cap']:
        mapa_regiao[val] = 'Capital'
    elif 'baixada' in val_norm:
        mapa_regiao[val] = 'Baixada Fluminense'
    elif 'serra' in val_norm:
        mapa_regiao[val] = 'Serra'
    elif 'verde' in val_norm or 'costa' in val_norm:
        mapa_regiao[val] = 'Costa Verde'

unidades['regiao'] = unidades['regiao'].map(mapa_regiao)
print("Valores após padronização:")
print(unidades['regiao'].value_counts())

### 4.2 Campo: `categoria_hotel`

**🔍 Problema identificado:** Variações de escrita — `'3*'`, `'Três Estrelas'`, `'4 Estrelas'`, `'cinco estrelas'`.

**✅ Ação:** Padronização para: `3 estrelas`, `4 estrelas`, `5 estrelas`.

**📌 Boa prática:** Campos ordinais devem ter formato consistente para permitir ordenação e comparação corretas.

In [ ]:
print("Valores únicos em categoria_hotel:")
print(unidades['categoria_hotel'].value_counts())

In [ ]:
mapa_cat = {}
for val in unidades['categoria_hotel'].dropna().unique():
    val_norm = val.strip().lower()
    if '3' in val_norm or 'três' in val_norm or 'tres' in val_norm:
        mapa_cat[val] = '3 estrelas'
    elif '4' in val_norm or 'quatro' in val_norm:
        mapa_cat[val] = '4 estrelas'
    elif '5' in val_norm or 'cinco' in val_norm:
        mapa_cat[val] = '5 estrelas'

unidades['categoria_hotel'] = unidades['categoria_hotel'].map(mapa_cat)
print("Valores após padronização:")
print(unidades['categoria_hotel'].value_counts())

### 4.3 Campo: `num_quartos_total` — Nulos

**🔍 Problema identificado:** 2 valores nulos neste campo.

**✅ Ação:** Consulta ao Departamento de Operações da NaraHoteis para obter os valores corretos. Não preencher com média, mediana ou estimativa.

**📌 Boa prática:** Nunca preencha nulos em campos operacionais críticos com estatísticas. O dado correto só pode vir da fonte.

In [ ]:
print("Unidades com num_quartos_total nulo:")
print(unidades[unidades['num_quartos_total'].isna()][['id_unidade', 'nome_unidade', 'num_quartos_total']])

In [ ]:
# Valores obtidos junto ao Departamento de Operações da NaraHoteis
# id_unidade 7 = NaraHoteis Petrópolis: 70 quartos
# id_unidade 10 = NaraHoteis Paraty: 65 quartos
unidades.loc[unidades['id_unidade'] == 7, 'num_quartos_total'] = 70
unidades.loc[unidades['id_unidade'] == 10, 'num_quartos_total'] = 65

print("Nulos após correção:", unidades['num_quartos_total'].isna().sum())

---
## 5. Pré-processamento — tipos_quarto.csv

In [ ]:
tipos_quarto

### 5.1 Campo: `descricao`

**🔍 Problema identificado:** Variações de caixa e espaços — `'STANDARD'`, `'superior '`, `'Deluxe'`.

**✅ Ação:** Padronização para: `Standard`, `Superior`, `Deluxe`, `Suite`, `Suíte Master`.

**📌 Boa prática:** Sempre aplique `strip()` para remover espaços antes e depois, e defina um padrão de capitalização antes de iniciar o tratamento.

In [ ]:
mapa_desc = {}
for val in tipos_quarto['descricao'].dropna().unique():
    val_norm = val.strip().lower()
    if 'standard' in val_norm:
        mapa_desc[val] = 'Standard'
    elif 'superior' in val_norm:
        mapa_desc[val] = 'Superior'
    elif 'deluxe' in val_norm:
        mapa_desc[val] = 'Deluxe'
    elif 'master' in val_norm:
        mapa_desc[val] = 'Suíte Master'
    elif 'suite' in val_norm or 'suíte' in val_norm:
        mapa_desc[val] = 'Suite'

tipos_quarto['descricao'] = tipos_quarto['descricao'].map(mapa_desc)
print(tipos_quarto['descricao'].value_counts())

### 5.2 Campo: `valor_diaria_base`

**🔍 Problema identificado:** Valor numérico armazenado como texto com símbolo de moeda — `'R$ 280,00'`.

**✅ Ação:** Remover o prefixo `'R$ '`, substituir vírgula por ponto e converter para numérico decimal.

**📌 Boa prática:** A ordem obrigatória é: **(1) limpar o conteúdo**, **(2) converter o tipo**. Nunca tente converter antes de tratar o conteúdo — isso gera erros.

In [ ]:
print("Antes:", tipos_quarto['valor_diaria_base'].tolist())

# (1) Limpar o conteúdo
tipos_quarto['valor_diaria_base'] = tipos_quarto['valor_diaria_base'].str.replace('R\$', '', regex=True)
tipos_quarto['valor_diaria_base'] = tipos_quarto['valor_diaria_base'].str.strip()
tipos_quarto['valor_diaria_base'] = tipos_quarto['valor_diaria_base'].str.replace('.', '', regex=False)
tipos_quarto['valor_diaria_base'] = tipos_quarto['valor_diaria_base'].str.replace(',', '.', regex=False)

# (2) Converter o tipo
tipos_quarto['valor_diaria_base'] = pd.to_numeric(tipos_quarto['valor_diaria_base'])

print("Depois:", tipos_quarto['valor_diaria_base'].tolist())
print("Tipo:", tipos_quarto['valor_diaria_base'].dtype)

### 5.3 Linha duplicada

**🔍 Problema identificado:** 1 linha duplicada identificada na auditoria inicial.

**✅ Ação:** Remoção da duplicata, mantendo apenas a primeira ocorrência.

**📌 Boa prática:** Em tabelas de dimensão, duplicatas contaminam todos os JOINs realizados com a tabela fato.

In [ ]:
print("Antes:", tipos_quarto.shape)
tipos_quarto = tipos_quarto.drop_duplicates().reset_index(drop=True)
print("Depois:", tipos_quarto.shape)
tipos_quarto

---
## 6. Pré-processamento — clientes.csv

In [ ]:
clientes.head()

### 6.1 Campo: `nome` — Espaços extras

**🔍 Problema identificado:** Espaços extras no início e/ou fim do campo.

**✅ Ação:** Aplicar remoção de espaços extras em todo o campo.

**📌 Boa prática:** Campos de texto livre acumulam espaços invisíveis que causam falhas silenciosas em buscas, filtros e joins. Sempre trate antes de usar.

In [ ]:
# Identificar registros com espaços extras
com_espaco = clientes[clientes['nome'] != clientes['nome'].str.strip()]
print(f"Registros com espaços extras em 'nome': {len(com_espaco)}")
print(com_espaco[['id_cliente', 'nome']].head())

clientes['nome'] = clientes['nome'].str.strip()
print("\nApós tratamento:", (clientes['nome'] != clientes['nome'].str.strip()).sum(), "registros com espaço")

### 6.2 Campo: `estado_origem`

**🔍 Problema identificado:** Variações — `'RJ'`, `'Rio de Janeiro'`, `'r.j.'`, `'rj'`.

**✅ Ação:** Padronização para sigla oficial de 2 letras em maiúsculo.

**📌 Boa prática:** Campos geográficos devem seguir nomenclatura oficial. Siglas de 2 letras são o padrão mais compacto e consistente para estados brasileiros.

In [ ]:
print("Variações encontradas em estado_origem:")
print(clientes['estado_origem'].value_counts())

In [ ]:
# Mapa de estados para sigla oficial
mapa_estados = {
    'Rio de Janeiro': 'RJ', 'r.j.': 'RJ', 'rj': 'RJ',
    'São Paulo': 'SP', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES',
}

def normalizar_estado(val):
    if pd.isna(val):
        return val
    val_strip = val.strip()
    return mapa_estados.get(val_strip, val_strip.upper()[:2] if len(val_strip) > 2 else val_strip.upper())

clientes['estado_origem'] = clientes['estado_origem'].apply(normalizar_estado)
print("\nApós padronização:")
print(clientes['estado_origem'].value_counts())

### 6.3 Campo: `tipo_cliente`

**🔍 Problema identificado:** Variações — `'PF'`, `'pessoa fisica'`, `'pf'`, `'PJ'`, `'corporativo'`.

**✅ Ação:** Padronização para: `Pessoa Física`, `Corporativo`.

**📌 Boa prática:** Campos de segmentação de clientes são críticos para análises comerciais. Qualquer inconsistência distorce a distribuição da base.

In [ ]:
print("Variações em tipo_cliente:")
print(clientes['tipo_cliente'].value_counts())

In [ ]:
mapa_tipo = {}
for val in clientes['tipo_cliente'].dropna().unique():
    val_norm = val.strip().lower()
    if any(x in val_norm for x in ['física', 'fisica', 'pf']):
        mapa_tipo[val] = 'Pessoa Física'
    elif any(x in val_norm for x in ['corporativo', 'pj', 'corp']):
        mapa_tipo[val] = 'Corporativo'

clientes['tipo_cliente'] = clientes['tipo_cliente'].map(mapa_tipo)
print("\nApós padronização:")
print(clientes['tipo_cliente'].value_counts())

### 6.4 Campo: `faixa_etaria` — Nulos

**🔍 Problema identificado:** Aproximadamente 10% de valores nulos.

**✅ Ação:** Manter os nulos. O preenchimento deste campo é opcional no cadastro.

**📌 Boa prática:** Nulos em campos opcionais de cadastro são esperados e válidos. Documentar o percentual e manter sem preenchimento artificial.

In [ ]:
nulos = clientes['faixa_etaria'].isna().sum()
pct = nulos / len(clientes) * 100
print(f"Nulos em faixa_etaria: {nulos} ({pct:.1f}%)")
print("Decisão: MANTER — campo de preenchimento opcional no cadastro.")

### 6.5 Campo: `id_cliente` — Registro duplicado com dados divergentes

**🔍 Problema identificado:** O `id_cliente` 47 aparece duas vezes com dados diferentes nas demais colunas.

**✅ Ação:** Analisar os dois registros, verificar qual é o mais completo ou recente e documentar a decisão.

**📌 Boa prática:** Este é o tipo mais crítico de duplicata — dois registros com a mesma chave e dados diferentes. A decisão de qual manter exige análise humana e deve ser justificada.

In [ ]:
# Identificar o registro duplicado
dup_id = clientes[clientes.duplicated(subset='id_cliente', keep=False)]
print("Registros com id_cliente duplicado:")
print(dup_id)

In [ ]:
# Análise: o segundo registro (linha com cidade_origem = 'Niterói') é mais completo
# e apresenta dados corporativos consistentes — decisão: manter o segundo, remover o primeiro

# Remover a primeira ocorrência do id 47
idx_remover = clientes[clientes['id_cliente'] == 47].index[0]
clientes = clientes.drop(index=idx_remover).reset_index(drop=True)

print(f"Shape após remoção: {clientes.shape}")
print(f"id_cliente 47 duplicado: {clientes.duplicated(subset='id_cliente').sum()} ocorrência(s)")

---
## 7. Pré-processamento — canais_venda.csv

In [ ]:
canais_venda

### 7.1 Campo: `nome_canal`

**🔍 Problema identificado:** Variações — `'ota'`, `'agencia'`.

**✅ Ação:** Padronização para: `Site Próprio`, `OTA`, `Telefone`, `Agência`.

**📌 Boa prática:** Mesmo em tabelas pequenas, a padronização é obrigatória. Inconsistências em tabelas de dimensão se propagam para toda a análise via JOIN.

In [ ]:
mapa_canal = {}
for val in canais_venda['nome_canal'].dropna().unique():
    val_norm = val.strip().lower()
    if 'site' in val_norm or 'próprio' in val_norm or 'proprio' in val_norm:
        mapa_canal[val] = 'Site Próprio'
    elif 'ota' in val_norm:
        mapa_canal[val] = 'OTA'
    elif 'telefone' in val_norm or 'tel' in val_norm:
        mapa_canal[val] = 'Telefone'
    elif 'agên' in val_norm or 'agen' in val_norm:
        mapa_canal[val] = 'Agência'

canais_venda['nome_canal'] = canais_venda['nome_canal'].map(mapa_canal)
print(canais_venda)

### 7.2 Campo: `comissao_pct`

**🔍 Problema identificado:** Valor numérico armazenado como texto com símbolo de percentual — `'12%'`, `'8%'`.

**✅ Ação:** Remover o símbolo `'%'` e converter para numérico decimal.

**📌 Boa prática:** A ordem obrigatória é: **(1) limpar o conteúdo**, **(2) converter o tipo**.

In [ ]:
# (1) Limpar
canais_venda['comissao_pct'] = canais_venda['comissao_pct'].str.replace('%', '', regex=False).str.strip()

# (2) Converter
canais_venda['comissao_pct'] = pd.to_numeric(canais_venda['comissao_pct'])

print(canais_venda)
print("\nTipo:", canais_venda['comissao_pct'].dtype)

---
## 8. Pré-processamento — funcionarios.csv

In [ ]:
funcionarios.head()

### 8.1 Campo: `salario` — Formato texto com aspas e separador de milhar

**🔍 Problema identificado:** Valor numérico armazenado como texto com aspas e separador de milhar — `'"3.500,00"'`.

**✅ Ação:** Remover aspas → remover separador de milhar (ponto) → substituir vírgula decimal por ponto → converter para numérico.

**📌 Boa prática:** Este é um padrão comum em exportações de sistemas legados brasileiros. A sequência de limpeza deve ser rigorosa: aspas → ponto de milhar → vírgula decimal → conversão.

In [ ]:
# (1) Limpar: remover aspas
funcionarios['salario'] = funcionarios['salario'].str.replace('"', '', regex=False)

# (2) Remover separador de milhar (ponto)
funcionarios['salario'] = funcionarios['salario'].str.replace('.', '', regex=False)

# (3) Substituir vírgula decimal por ponto
funcionarios['salario'] = funcionarios['salario'].str.replace(',', '.', regex=False)

# (4) Converter para numérico
funcionarios['salario'] = pd.to_numeric(funcionarios['salario'], errors='coerce')

print("Tipo após conversão:", funcionarios['salario'].dtype)
print("Nulos (mantidos):", funcionarios['salario'].isna().sum())

### 8.2 Campo: `salario` — Valor negativo (id_funcionario 11)

**🔍 Problema identificado:** Salário negativo `-2800.00` no registro do id_funcionario 11.

**✅ Ação:** Substituição pelo valor correto conforme **Comunicado Oficial COM-2025-047**: R$ 1.950,00.

**📌 Boa prática:** Salários negativos são erros de entrada. Corrija sempre com base em fonte oficial.

In [ ]:
print("Antes:", funcionarios.loc[funcionarios['id_funcionario'] == 11, 'salario'].values)
funcionarios.loc[funcionarios['id_funcionario'] == 11, 'salario'] = 1950.00
print("Depois:", funcionarios.loc[funcionarios['id_funcionario'] == 11, 'salario'].values)

### 8.3 Campo: `salario` — Outlier extremo (id_funcionario 26)

**🔍 Problema identificado:** Valor extremo `999999.00` no registro do id_funcionario 26, detectável via IQR.

**✅ Ação:** Substituição pelo valor correto conforme **Comunicado Oficial COM-2025-047**: R$ 2.300,00.

**📌 Boa prática:** Outliers em campos salariais devem ser investigados antes de qualquer decisão. Confirme com a fonte antes de remover ou substituir.

In [ ]:
# Verificando via IQR antes de corrigir
sal = funcionarios['salario'].dropna()
q1 = np.percentile(sal, 25)
q3 = np.percentile(sal, 75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
print(f"Limite superior: {limite_superior:.2f}")
print("\nSalários acima do limite:")
print(funcionarios[funcionarios['salario'] > limite_superior][['id_funcionario', 'nome', 'cargo', 'salario']])

In [ ]:
funcionarios.loc[funcionarios['id_funcionario'] == 26, 'salario'] = 2300.00
print("Correto:", funcionarios.loc[funcionarios['id_funcionario'] == 26, 'salario'].values)

### 8.4 Campo: `salario` — Nulos

**🔍 Problema identificado:** Aproximadamente 5 registros com salário nulo.

**✅ Ação:** Manter os nulos após a conversão de tipo. Não preencher com média ou mediana.

**📌 Boa prática:** Nulos em campos salariais podem ter diversas origens. Nunca preencha sem confirmação da fonte.

In [ ]:
print(f"Nulos em salario: {funcionarios['salario'].isna().sum()}")
print("Decisão: MANTER — origem desconhecida, requer confirmação da fonte.")

### 8.5 Campo: `cargo`

**🔍 Problema identificado:** Variações de caixa e espaços — `'RECEPCIONISTA'`, `'camareira'`, `'Gerente '`.

**✅ Ação:** Padronizar caixa (inicial maiúscula) e remover espaços extras.

**📌 Boa prática:** Campos de cargo são usados em agrupamentos e filtros de RH. Inconsistências distorcem headcounts e análises por função.

In [ ]:
funcionarios['cargo'] = funcionarios['cargo'].str.strip().str.title()
print(funcionarios['cargo'].value_counts())

### 8.6 Campo: `departamento`

**🔍 Problema identificado:** Abreviações inconsistentes — `'Admin.'`, `'A&B'`, `'Gov.'`.

**✅ Ação:** Padronização para os nomes completos oficiais.

**📌 Boa prática:** Abreviações são ambíguas e devem ser evitadas em bases de dados. Sempre use o nome completo padronizado.

In [ ]:
mapa_depto = {}
for val in funcionarios['departamento'].dropna().unique():
    val_norm = val.strip().lower()
    if 'admin' in val_norm:
        mapa_depto[val] = 'Administração'
    elif 'a&b' in val_norm or 'alim' in val_norm or 'bebid' in val_norm:
        mapa_depto[val] = 'Alimentos e Bebidas'
    elif 'gov' in val_norm:
        mapa_depto[val] = 'Governança'
    elif 'recep' in val_norm:
        mapa_depto[val] = 'Recepção'
    elif 'manu' in val_norm:
        mapa_depto[val] = 'Manutenção'
    elif 'segu' in val_norm:
        mapa_depto[val] = 'Segurança'
    elif 'fin' in val_norm or 'rh' in val_norm or 'rec' in val_norm:
        mapa_depto[val] = 'Financeiro'

funcionarios['departamento'] = funcionarios['departamento'].map(mapa_depto)
print(funcionarios['departamento'].value_counts())

---
## 9. Exportação das Bases Tratadas

In [ ]:
reservas.to_csv('../dados/tratados/reservas_tratado.csv', sep=';', index=False, encoding='utf-8')
unidades.to_csv('../dados/tratados/unidades_tratado.csv', sep=';', index=False, encoding='utf-8')
tipos_quarto.to_csv('../dados/tratados/tipos_quarto_tratado.csv', sep=';', index=False, encoding='utf-8')
clientes.to_csv('../dados/tratados/clientes_tratado.csv', sep=';', index=False, encoding='utf-8')
canais_venda.to_csv('../dados/tratados/canais_venda_tratado.csv', sep=';', index=False, encoding='utf-8')
funcionarios.to_csv('../dados/tratados/funcionarios_tratado.csv', sep=';', index=False, encoding='utf-8')

print("✅ Bases exportadas com sucesso para a pasta dados/tratados/")

---
## 10. Análise Estatística

### 10.1 Preparação — Cálculo de Receita e RevPAR

O **RevPAR** (Revenue Per Available Room) é uma métrica central na hotelaria:

> **RevPAR = Receita Total ÷ Número de Quartos Disponíveis**

Calculamos a receita por reserva cruzando `qtd_diarias` com `valor_diaria_base` de `tipos_quarto`.


In [ ]:
# Merge reservas + tipos_quarto para obter valor_diaria_base
df = reservas.merge(tipos_quarto[['id_tipo_quarto', 'valor_diaria_base']], on='id_tipo_quarto', how='left')

# Calcular receita por reserva (apenas reservas com status válido e qtd_diarias positiva)
df = df[df['status_reserva'].isin(['Confirmada', 'Concluída'])]
df['receita'] = df['qtd_diarias'] * df['valor_diaria_base']

# Merge com unidades para agregar por unidade
df = df.merge(unidades[['id_unidade', 'nome_unidade', 'regiao', 'num_quartos_total']], on='id_unidade', how='left')

# RevPAR por unidade
revpar = df.groupby(['id_unidade', 'nome_unidade', 'regiao', 'num_quartos_total'])['receita'].sum().reset_index()
revpar.columns = ['id_unidade', 'nome_unidade', 'regiao', 'num_quartos_total', 'receita_total']
revpar['revpar'] = revpar['receita_total'] / revpar['num_quartos_total']
revpar = revpar.sort_values(by='revpar').reset_index(drop=True)
revpar

### 10.2 Classificação das Variáveis

In [ ]:
print(Variáveis QUALITATIVAS NOMINAIS:  - regiao, categoria_hotel, status_reserva, forma_pagamento  - nome_canal, tipo_cliente, estado_origem, cargo, departamentoVariáveis QUALITATIVAS ORDINAIS:  - categoria_hotel (3 < 4 < 5 estrelas)  - faixa_etaria (18-25 < 26-35 < ... < 65+)Variáveis QUANTITATIVAS DISCRETAS:  - id_reserva, id_unidade, num_hospedes, qtd_diarias  - num_quartos_total, id_funcionarioVariáveis QUANTITATIVAS CONTÍNUAS:  - valor_diaria_base, receita, revpar, comissao_pct, salario, avaliacao_hospede)

### 10.3 Medidas de Tendência Central — Valor da Diária

In [ ]:
diarias = np.array(df['valor_diaria_base'])

media   = np.mean(diarias)
mediana = np.median(diarias)

print(f"Média:   R$ {media:.2f}")
print(f"Mediana: R$ {mediana:.2f}")
print()
if abs(media - mediana) / mediana > 0.05:
    print("A média e a mediana apresentam diferença relevante.")
    print("Isso indica assimetria na distribuição — a mediana é mais representativa.")
else:
    print("Média e mediana próximas — distribuição relativamente simétrica.")

### 10.4 Quartis do Valor da Diária por Região

In [ ]:
for regiao in sorted(df['regiao'].dropna().unique()):
    vals = np.array(df[df['regiao'] == regiao]['valor_diaria_base'])
    q1 = np.percentile(vals, 25)
    q2 = np.percentile(vals, 50)
    q3 = np.percentile(vals, 75)
    print(f"{regiao}")
    print(f"  Q1: R$ {q1:.2f} | Q2 (mediana): R$ {q2:.2f} | Q3: R$ {q3:.2f}")
    print()

### 10.5 Dispersão — Variância, Desvio Padrão e Coeficiente de Variação

In [ ]:
variancia = np.var(diarias)
desvio_padrao = np.std(diarias)
cv = (desvio_padrao / media) * 100

print(f"Variância:              {variancia:.2f}")
print(f"Desvio padrão:         R$ {desvio_padrao:.2f}")
print(f"Coeficiente de variação: {cv:.1f}%")

### 10.6 Detecção de Outliers via IQR — RevPAR por Unidade

In [ ]:
vals_revpar = np.array(revpar['revpar'])

Q1 = np.percentile(vals_revpar, 25)
Q2 = np.percentile(vals_revpar, 50)
Q3 = np.percentile(vals_revpar, 75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f} | Q2: {Q2:.2f} | Q3: {Q3:.2f} | IQR: {IQR:.2f}")
print(f"Limite inferior: {limite_inf:.2f} | Limite superior: {limite_sup:.2f}")
print()

outliers = revpar[(revpar['revpar'] < limite_inf) | (revpar['revpar'] > limite_sup)]
print("Unidades com RevPAR outlier:")
print(outliers[['nome_unidade', 'regiao', 'receita_total', 'revpar']])

### 10.7 Overbooking — Unidades operando além da capacidade

In [ ]:
# Reservas confirmadas por unidade por mês
reservas_conf = reservas[reservas['status_reserva'] == 'Confirmada'].copy()
reservas_conf['periodo'] = reservas_conf['data_checkin'].str[:7]

por_periodo = reservas_conf.groupby(['id_unidade', 'periodo']).size().reset_index(name='total_reservas')
por_periodo = por_periodo.merge(unidades[['id_unidade', 'nome_unidade', 'num_quartos_total']], on='id_unidade')
por_periodo['overbooking'] = por_periodo['total_reservas'] > por_periodo['num_quartos_total']

ob = por_periodo[por_periodo['overbooking']].sort_values(by='total_reservas', ascending=False)
print(f"Períodos com overbooking: {len(ob)}")
print()
print(ob[['nome_unidade', 'periodo', 'total_reservas', 'num_quartos_total']].to_string(index=False))

### 10.8 Correlação de Pearson — RevPAR × Avaliação Média

In [ ]:
# Avaliação média por unidade
aval_media = df[df['avaliacao_hospede'].notna()].groupby('id_unidade')['avaliacao_hospede'].mean().reset_index()
aval_media.columns = ['id_unidade', 'avaliacao_media']

# Merge com revpar
analise = revpar.merge(aval_media, on='id_unidade')

x = np.array(analise['avaliacao_media'])
y = np.array(analise['revpar'])

correlacao = np.corrcoef(x, y)[0, 1]
print(f"Correlação de Pearson (avaliação média × RevPAR): r = {correlacao:.4f}")
print()
if abs(correlacao) >= 0.8:
    print("Correlação FORTE — justifica análise preditiva via regressão linear.")
elif abs(correlacao) >= 0.5:
    print("Correlação MODERADA — regressão possível com cautela.")
else:
    print("Correlação FRACA — regressão linear não é recomendada.")

### 10.9 Regressão Linear — Estimativa de RevPAR a partir da Avaliação Média

In [ ]:
from sklearn.linear_model import LinearRegression

X = x.reshape(-1, 1)
Y = y

modelo = LinearRegression()
modelo.fit(X, Y)

a = modelo.coef_[0]
b = modelo.intercept_
r2 = modelo.score(X, Y)

print(f"Equação: RevPAR = {a:.2f} × Avaliação + {b:.2f}")
print(f"R²: {r2:.4f}")
print()
print("Interpretação:")
print(f"  Para cada ponto a mais na avaliação média, o RevPAR aumenta R$ {a:.2f}")
print(f"  O modelo explica {r2*100:.1f}% da variação no RevPAR observado.")
print()
print("Atenção: este modelo identifica TENDÊNCIAS com base em dados históricos.")
print("Não representa uma previsão do futuro.")

### 10.10 Variabilidade por Região

In [ ]:
print("Coeficiente de Variação do RevPAR por Região:")
print()
for regiao in sorted(revpar['regiao'].dropna().unique()):
    vals = revpar[revpar['regiao'] == regiao]['revpar'].values
    if len(vals) > 1:
        cv_reg = (np.std(vals) / np.mean(vals)) * 100
        print(f"  {regiao}: CV = {cv_reg:.1f}%")
    else:
        print(f"  {regiao}: amostra insuficiente para CV")

---
## 11. Painel — Matplotlib

Painel com 2 linhas × 2 colunas respondendo perguntas de negócio da diretoria NaraHoteis.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("NaraHoteis — Painel Gerencial de Performance", fontsize=16, fontweight='bold', color='#2D5A27', y=1.01)

VERDE     = '#2D5A27'
VERDE_M   = '#4A7C3F'
DOURADO   = '#8B6914'
CINZA     = '#64748B'

# ── Quadrante 1 (topo esquerda): Boxplot RevPAR por Região ──────────────────
ax1 = axes[0, 0]
regioes_ord = sorted(revpar['regiao'].dropna().unique())
dados_box = [revpar[revpar['regiao'] == r]['revpar'].values for r in regioes_ord]
bp = ax1.boxplot(dados_box, vert=True, patch_artist=True, showmeans=True,
                 meanprops=dict(marker='D', markerfacecolor=DOURADO, markeredgecolor=DOURADO, markersize=7))
for patch in bp['boxes']:
    patch.set_facecolor('#E8F0E6')
    patch.set_edgecolor(VERDE)
for median in bp['medians']:
    median.set_color(VERDE)
ax1.set_xticks(range(1, len(regioes_ord)+1))
ax1.set_xticklabels(regioes_ord, rotation=15, ha='right', fontsize=9)
ax1.set_title("Distribuição do RevPAR por Região", fontweight='bold', color=VERDE)
ax1.set_ylabel("RevPAR (R$)")
ax1.grid(axis='y', alpha=0.3)

# ── Quadrante 2 (topo direita): Boxplot Avaliação por Tipo de Quarto ────────
ax2 = axes[0, 1]
df_tq = df.merge(tipos_quarto[['id_tipo_quarto', 'descricao']], on='id_tipo_quarto', how='left')
tipos_ord = ['Standard', 'Superior', 'Deluxe', 'Suite', 'Suíte Master']
dados_aval = [df_tq[df_tq['descricao'] == t]['avaliacao_hospede'].dropna().values for t in tipos_ord]
bp2 = ax2.boxplot(dados_aval, vert=True, patch_artist=True, showmeans=True,
                  meanprops=dict(marker='D', markerfacecolor=DOURADO, markeredgecolor=DOURADO, markersize=7))
for patch in bp2['boxes']:
    patch.set_facecolor('#E8F0E6')
    patch.set_edgecolor(VERDE_M)
for median in bp2['medians']:
    median.set_color(VERDE_M)
ax2.set_xticks(range(1, len(tipos_ord)+1))
ax2.set_xticklabels(tipos_ord, rotation=15, ha='right', fontsize=9)
ax2.set_title("Avaliação dos Hóspedes por Tipo de Quarto", fontweight='bold', color=VERDE)
ax2.set_ylabel("Avaliação (1–10)")
ax2.grid(axis='y', alpha=0.3)

# ── Quadrante 3 (baixo esquerda): RevPAR por Unidade (barras) ───────────────
ax3 = axes[1, 0]
rev_sorted = revpar.sort_values(by='revpar')
cores = [DOURADO if v < (Q1 - 1.5 * IQR + Q1) / 2 else VERDE_M for v in rev_sorted['revpar']]
# Simplificando: bottom 2 em dourado (destaque), resto em verde
cores = [DOURADO if i < 2 else VERDE_M for i in range(len(rev_sorted))]
ax3.barh(rev_sorted['nome_unidade'].str.replace('NaraHoteis ', ''), rev_sorted['revpar'],
         color=cores, edgecolor='white')
ax3.set_title("RevPAR por Unidade", fontweight='bold', color=VERDE)
ax3.set_xlabel("RevPAR (R$)")
ax3.axvline(x=float(np.mean(rev_sorted['revpar'])), color=CINZA, linestyle='--', linewidth=1, label='Média')
ax3.legend(fontsize=8)
ax3.grid(axis='x', alpha=0.3)

# ── Quadrante 4 (baixo direita): Dispersão Avaliação × RevPAR + Regressão ──
ax4 = axes[1, 1]
ax4.scatter(analise['avaliacao_media'], analise['revpar'], color=VERDE_M, s=80, zorder=5, label='Unidades')
x_linha = np.linspace(analise['avaliacao_media'].min(), analise['avaliacao_media'].max(), 100)
y_linha = a * x_linha + b
ax4.plot(x_linha, y_linha, color=DOURADO, linewidth=2, label=f'Regressão (r={correlacao:.2f})')
for _, row in analise.iterrows():
    ax4.annotate(row['nome_unidade'].replace('NaraHoteis ', ''),
                 (row['avaliacao_media'], row['revpar']),
                 textcoords='offset points', xytext=(5, 3), fontsize=7, color=CINZA)
ax4.set_title("Avaliação Média × RevPAR por Unidade", fontweight='bold', color=VERDE)
ax4.set_xlabel("Avaliação Média")
ax4.set_ylabel("RevPAR (R$)")
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../dados/tratados/painel_narahoteis.png', dpi=150, bbox_inches='tight')
plt.show()